# 07 — Results Summary and Capstone Handoff
**PentAI | Goat and Cattle Disease Detection**

Assemble a reproducible summary of **approved source splits, model selection and held-out evaluation** from previous notebooks. This notebook does not train a model and does not replace a written capstone report. It also generates a handoff checklist for team review.

## 1. Mount Google Drive

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Local Python session detected. Edit PROJECT_ROOT in the setup cell to your local dataset folder.')

## 2. Configuration and reusable data utilities
Change `RUN_OWNER` in the first cell. For local Jupyter, change `PROJECT_ROOT` and `OUTPUT_ROOT`.

In [ ]:
from pathlib import Path
import os, json, zipfile, shutil, hashlib, random
import numpy as np
import pandas as pd
from PIL import Image
import tensorflow as tf
from IPython.display import display

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
PROJECT_ROOT = Path('/content/drive/MyDrive/Capstone Project')
DATASETS_DIR = PROJECT_ROOT / 'Datasets'
SPLIT_DIR = PROJECT_ROOT / 'outputs' / 'prepared'
WORK_DIR = Path('/content/pentai_data')
# Every member must set their own name: output is private to their MyDrive by default.
RUN_OWNER = 'CHANGE_TO_YOUR_NAME'
if RUN_OWNER == 'CHANGE_TO_YOUR_NAME':
    raise ValueError('Set RUN_OWNER to your own name before running. Do not write to shared team outputs.')
OUTPUT_ROOT = Path('/content/drive/MyDrive/PentAI_Individual_Outputs') / RUN_OWNER
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
ZIP_NAMES = {'cattle': ['Cow data.zip'],
             'goat': ['Goat Pox Detection.v1i.tensorflow.zip', 'Goat Pox Detection.v1i.tensorflow(1).zip']}
IMG_SIZE=(224,224)
BATCH_SIZE=32
AUTOTUNE=tf.data.AUTOTUNE
print('TensorFlow:', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# Keep the original ZIP files unchanged. Extract copies only into temporary Colab storage.
def safe_extract_zip(zip_path, target):
    target.mkdir(parents=True, exist_ok=True)
    done = target / '.extraction_complete'
    if done.exists(): return target
    with zipfile.ZipFile(zip_path) as z:
        base = target.resolve()
        for member in z.infolist():
            dest = (target / member.filename).resolve()
            if not dest.is_relative_to(base):
                raise ValueError(f'Unsafe filename in archive: {member.filename}')
        z.extractall(target)
    done.touch()
    return target

def ensure_images(animal):
    target = WORK_DIR / animal
    if (target / '.extraction_complete').exists(): return target
    zips = [DATASETS_DIR / x for x in ZIP_NAMES[animal]]
    found = next((z for z in zips if z.exists()), None)
    if found is None:
        raise FileNotFoundError(f'{animal}: expected one of {zips}. Change ZIP_NAMES if needed.')
    print('Extracting:', found.name)
    return safe_extract_zip(found, target)

def make_path_index(root):
    # Robust against ZIPs that introduce an extra top-level directory.
    paths = [p for p in root.rglob('*') if p.is_file() and p.suffix.lower() in {'.jpg','.jpeg','.png','.bmp','.gif'}]
    by_tail = {}
    for p in paths:
        parts = p.relative_to(root).as_posix().split('/')
        for i in range(len(parts)):
            key = '/'.join(parts[i:]).casefold()
            by_tail.setdefault(key, set()).add(str(p))
    return by_tail

def resolve_manifest_paths(df, root):
    index = make_path_index(root)
    resolved = []
    for relative in df['relative_path'].astype(str):
        key = relative.replace('\\','/').lstrip('/').casefold()
        matches = index.get(key, set())
        if len(matches) != 1:
            raise ValueError(f'Image path must resolve uniquely ({len(matches)} matches): {relative}')
        resolved.append(next(iter(matches)))
    result = df.copy()
    result['absolute_path'] = resolved
    return result

def load_splits(animal, partitions=('train','valid')):
    items=[]
    for part in partitions:
        csv_path=SPLIT_DIR / f'{animal}_{part}.csv'
        if not csv_path.exists(): raise FileNotFoundError(f'Missing Notebook 02 output: {csv_path}')
        frame=pd.read_csv(csv_path, keep_default_na=False)
        if not {'relative_path','class','split'}.issubset(frame.columns):
            raise ValueError(f'{csv_path.name} lacks a required column')
        if frame.empty or not frame['split'].eq(part).all():
            raise ValueError(f'{csv_path.name} is empty or contains the wrong partition')
        if not frame['relative_path'].is_unique: raise ValueError(f'Duplicate paths in {csv_path.name}')
        frame['class'] = frame['class'].astype(str)
        items.append(frame)
    combined=pd.concat(items,ignore_index=True)
    if combined['relative_path'].duplicated().any(): raise ValueError('Image appears in more than one split')
    if 'sha256' in combined and (combined.groupby('sha256')['split'].nunique()>1).any():
        raise ValueError('Exact hash leakage found between splits')
    if 'similarity_group' in combined and (combined.groupby('similarity_group')['split'].nunique()>1).any():
        raise ValueError('Possible similar-image leakage found between splits')
    return resolve_manifest_paths(combined,ensure_images(animal))

def check_approval():
    # The team must first approve the Notebook 01/02 reviews; never infer approval from CSV existence.
    if not TEAM_APPROVED_SPLITS:
        raise RuntimeError('Notebook 02 splits still require team approval. Set TEAM_APPROVED_SPLITS=True ONLY after sign-off.')

def make_ds(df, classes, training=False):
    labels = {label:i for i,label in enumerate(classes)}
    unknown = set(df['class']) - set(labels)
    if unknown: raise ValueError(f'Unexpected labels: {sorted(unknown)}')
    p = df['absolute_path'].astype(str).to_numpy()
    y = np.array([labels[x] for x in df['class']],dtype=np.int32)
    ds=tf.data.Dataset.from_tensor_slices((p,y))
    def read_image(path,label):
        im = tf.io.decode_image(tf.io.read_file(path),channels=3,expand_animations=False)
        im.set_shape((None,None,3))
        im=tf.image.resize(tf.cast(im,tf.float32),IMG_SIZE)
        return im,label
    if training: ds=ds.shuffle(len(df),seed=SEED,reshuffle_each_iteration=True)
    return ds.map(read_image,num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

def save_metadata(path, meta):
    path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(json.dumps(meta,indent=2),encoding='utf-8')

AUG = "tf.keras.Sequential([tf.keras.layers.RandomFlip('horizontal'),tf.keras.layers.RandomRotation(0.08),tf.keras.layers.RandomZoom(0.10)],name='train_only_augmentation')"

## 3. Read existing outputs

In [ ]:
summary_file=SPLIT_DIR/'split_summary.json'
selection_file=OUTPUT_ROOT/'models'/'selection.json'
metrics_file=OUTPUT_ROOT/'evaluation'/'final_metrics.csv'
for f in (summary_file,selection_file,metrics_file):
    if not f.is_file(): raise FileNotFoundError(f'Missing prerequisite output: {f}')
summary=json.loads(summary_file.read_text())
selection=json.loads(selection_file.read_text())
metrics=pd.read_csv(metrics_file)
print('Notebook 02 approval field:',summary.get('approval_status','not available'))
print('Training selection:',{a:record['variant'] for a,record in selection.items()})
display(metrics)

## 4. Write a Markdown project-results report

In [ ]:
from datetime import datetime,timezone
report_lines=[
'# PentAI — Goat and Cattle Disease Detection: Results',
'',f'Generated: {datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")}',
'', '## Scope', 'Two separate image classifiers for goats and cattle. Poultry is excluded.',
'', '## Method',
'- Notebook 01: dataset audits, exclusions and exploratory analysis.',
'- Notebook 02: split validation, near-duplicate review and split manifests.',
'- Notebook 03: from-scratch convolutional baseline.',
'- Notebook 04: MobileNetV2 transfer learning and validation-only model choice.',
'- Notebook 05: held-out test evaluation of the frozen winners.',
'- Notebook 06: per-species demonstration on new images.',
'', '## Held-out results',
'', '| Animal | Selected model | Test N | Accuracy | Macro F1 | Balanced accuracy |',
'|---|---|---:|---:|---:|---:|']
for row in metrics.to_dict(orient='records'):
    report_lines.append(f"| {row['animal']} | {row['selected_variant']} | {int(row['test_samples'])} | {row['accuracy']:.3f} | {row['macro_f1']:.3f} | {row['balanced_accuracy']:.3f} |")
report_lines += ['', '## Important limitations',
'- Dataset size, collection setting and class imbalance constrain generalization.',
'- Duplicate and annotation review depends on human decisions; thresholds alone are insufficient.',
'- Confidence scores are not necessarily calibrated probabilities.',
'- This research prototype is not validated as a veterinary diagnostic device.',
'', '## Team sign-off before final submission',
'- [ ] Confirm Notebook 01 exclusion audit decisions and provenance.',
'- [ ] Sign off Notebook 02 class labels, group decisions and partitions.',
'- [ ] Verify each species has independent results and confusion matrices.',
'- [ ] Confirm held-out test was evaluated only after selecting models.',
'- [ ] Document task ownership, dataset licenses, code versions and limitations.',
'- [ ] Review report and demo with the full team.',
'', '## Reproducibility',
'Run notebooks in order: 01 -> 02 -> 03 -> 04 -> 05 -> 06 -> 07.',
'Raw ZIP files and patient/animal-owner sensitive information are not included in GitHub.']
report_path=OUTPUT_ROOT/'reports'/'capstone_results.md'
report_path.parent.mkdir(parents=True,exist_ok=True)
report_path.write_text('\n'.join(report_lines)+'\n',encoding='utf-8')
print(report_path)
print(report_path.read_text())

## 5. GitHub / ClickUp handoff
Commit `.ipynb` notebooks, a reviewed README and small CSV/Markdown reports; avoid ZIP images, Drive credentials, and large trained model binaries. Close each ClickUp task only after its notebook, outputs and reviewer sign-off are recorded. The capstone report should credit **Prepared by Dan Wamatu** where the team intends that attribution.